**Install Required Packages**

In [17]:
!pip install google-cloud-bigquery google-cloud-storage google-auth


**Import libraries and initialize client instances**


In [18]:
import os
import asyncio
import google.auth

os.environ['GOOGLE_CLOUD_PROJECT']

GOOGLE_CLOUD_PROJECT = "qwiklabs-gcp-01-c39c93a00054"
GOOGLE_CLOUD_LOCATION = "us-central1"
MODEL_NAME = "gemini-2.5-flash"

os.environ['GOOGLE_CLOUD_PROJECT'] = GOOGLE_CLOUD_PROJECT
os.environ['GOOGLE_CLOUD_LOCATION'] = GOOGLE_CLOUD_LOCATION
os.environ['MODEL_NAME'] = MODEL_NAME

google.auth.default()

(<google.auth.compute_engine.credentials.Credentials at 0x7adbf53d6510>,
 'qwiklabs-gcp-01-c39c93a00054')

In [22]:
!gcloud auth application-default login


You are running on a Google Compute Engine virtual machine.
The service credentials associated with this virtual machine
will automatically be used by Application Default
Credentials, so it is not necessary to use this command.

If you decide to proceed anyway, your user credentials may be visible
to others with access to this virtual machine. Are you sure you want
to authenticate with your personal account?

Do you want to continue (Y/n)?  Y

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=RNNb9pqHw4TQaWIlFKJP4NnrGOPDuz&prompt=consent&token_

In [23]:
os.getenv("GOOGLE_CLOUD_PROJECT")
os.getenv("GOOGLE_CLOUD_LOCATION")

'us-central1'

In [27]:
from google.cloud import bigquery
from google.cloud import storage

# Initialize BigQuery and Storage clients
bigquery_client = bigquery.Client()
storage_client = storage.Client()

# --- Configuration for your data load ---
# Replace with your project ID, dataset ID, table ID, and GCS bucket details
project_id = "qwiklabs-gcp-01-c39c93a00054"
location = "us-central1"
dataset_id = "fraud_detection"
raw_table_id = "fraud_data_raw"
gold_table_id = "fraud_training_data"
source_bucket_name = "gs://labs.roitraining.com/data-to-ai-workshop"
source_file_name = "gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv"

**Create Dataset**

In [25]:
sql_ddl_dataset_statement = f"""
CREATE SCHEMA IF NOT EXISTS `{project_id}.{dataset_id}`
OPTIONS(
    location = '{location}',
    default_table_expiration_days=14
);
"""

# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(sql_ddl_dataset_statement)

# Wait for the job to complete
query_job.result()

print(f"Dataset '{dataset_id}' created or already exists in project '{project_id}' at location '{location}'.")


Dataset 'fraud_detection' created or already exists in project 'qwiklabs-gcp-01-c39c93a00054' at location 'us-central1'.


**Create Table**

In [26]:
sql_ddl_statement = f"""
DROP TABLE IF EXISTS `{project_id}.{dataset_id}.{raw_table_id}`;

CREATE TABLE `{project_id}.{dataset_id}.{raw_table_id}` (
    Applicant_ID STRING,
    Age INTEGER,
    Employment_Status STRING,
    Income NUMERIC,
    Number_of_Dependents INTEGER,
    Amount_Requested NUMERIC,
    Previous_Assistance_Received BOOLEAN,
    Previous_Assistance_Date DATE,
    Supporting_Doc_Verified BOOLEAN,
    Application_Frequency_Last_Year INTEGER,
    IP_Address STRING,
    Device_Type STRING,
    Application_Date DATE,
    Fraudulent BOOLEAN
);
"""

# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(sql_ddl_statement)

# Wait for the job to complete
query_job.result()

print(f"Table '{raw_table_id}' created or replaced in dataset '{dataset_id}'.")


Table 'fraud_data_raw' created or replaced in dataset 'fraud_detection'.


**Load data into Big Query Table**

In [30]:
sql_ddl_statement = f"""
LOAD DATA INTO `{project_id}.{dataset_id}.{raw_table_id}`
(
    Applicant_ID STRING,
    Age INTEGER,
    Employment_Status STRING,
    Income NUMERIC,
    Number_of_Dependents INTEGER,
    Amount_Requested NUMERIC,
    Previous_Assistance_Received BOOLEAN,
    Previous_Assistance_Date DATE,
    Supporting_Doc_Verified BOOLEAN,
    Application_Frequency_Last_Year INTEGER,
    IP_Address STRING,
    Device_Type STRING,
    Application_Date DATE,
    Fraudulent BOOLEAN
)
FROM FILES (
format = 'CSV',
field_delimiter = ',',
max_bad_records = 10,
uris = ['{source_file_name}']);
"""
# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(sql_ddl_statement)

# Wait for the job to complete
query_job.result()

print(f"Table '{raw_table_id}' loaded with raw data from '{source_file_name}'.")

Table 'fraud_data_raw' loaded with raw data from 'gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv'.


In [50]:
df = bigquery_client.list_rows(f"{project_id}.{dataset_id}.{raw_table_id}").to_dataframe()
print(df.head())


  Applicant_ID  Age Employment_Status           Income  Number_of_Dependents  \
0          217   65        Unemployed  28984.000000000                     4   
1          226   54     Self-Employed             0E-9                     1   
2          240   26     Self-Employed  64477.000000000                     5   
3          252   28        Unemployed  28576.000000000                     4   
4          266   43          Employed  44930.000000000                     5   

  Amount_Requested  Previous_Assistance_Received Previous_Assistance_Date  \
0   5872.000000000                         False                      NaT   
1   6631.000000000                         False                      NaT   
2   8612.000000000                         False                      NaT   
3   2951.000000000                         False                      NaT   
4   2324.000000000                         False                      NaT   

   Supporting_Doc_Verified  Application_Frequency_Last_Y

In [51]:
# Transform the data and store it in golden layer
# a. One-hot encode the Employment_Status and Device_Type fields
# b. Break the age field into bins (e.g., 18–24, 25–34, 35–44, etc.), and one-hot encode the bins
# c. Create a field called Income-to-Amount-Requested which is the ratio of those fields
# d. Create a calculated field called Time_Since_Previous_Assistance
# e. Change True/False fields to 0s and 1s

import pandas as pd
from decimal import Decimal, ROUND_HALF_UP

def precise_round(val):
    return Decimal(str(val)).quantize(Decimal('0.00'), rounding=ROUND_HALF_UP)


# a. One-hot encode the Employment_Status and Device_Type fields
df_encoded_employment = pd.get_dummies(df['Employment_Status'], prefix='Employment')
df_encoded_device = pd.get_dummies(df['Device_Type'], prefix='Device')
df = pd.concat([df, df_encoded_employment, df_encoded_device], axis=1)

# b. Break the age field into bins and one-hot encode
age_bins = [18, 25, 35, 45, 55, 65, float('inf')]
age_labels = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
df['Age_Bin'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels, right=False)
df_encoded_age = pd.get_dummies(df['Age_Bin'], prefix='Age')
df = pd.concat([df, df_encoded_age], axis=1)

# c. Create a field called Income-to-Amount-Requested
df['Income_to_Amount_Requested'] = (df['Income'] / df['Amount_Requested']).apply(precise_round)

# d. Create a calculated field called Time_Since_Previous_Assistance
# Assuming 'Application_Date' and 'Previous_Assistance_Date' are datetime objects
df['Previous_Assistance_Date'] = pd.to_datetime(df['Previous_Assistance_Date'])
df['Time_Since_Previous_Assistance'] = (df['Previous_Assistance_Date']- pd.Timestamp.now()).dt.days

# e. Change True/False fields to 0s and 1s
bool_cols = df.select_dtypes(include='bool').columns
for col in bool_cols:
    df[col] = df[col].astype(int)

# Display the modified DataFrame
print(df.head())

  Applicant_ID  Age Employment_Status           Income  Number_of_Dependents  \
0          217   65        Unemployed  28984.000000000                     4   
1          226   54     Self-Employed             0E-9                     1   
2          240   26     Self-Employed  64477.000000000                     5   
3          252   28        Unemployed  28576.000000000                     4   
4          266   43          Employed  44930.000000000                     5   

  Amount_Requested  Previous_Assistance_Received Previous_Assistance_Date  \
0   5872.000000000                             0                      NaT   
1   6631.000000000                             0                      NaT   
2   8612.000000000                             0                      NaT   
3   2951.000000000                             0                      NaT   
4   2324.000000000                             0                      NaT   

   Supporting_Doc_Verified  Application_Frequency_Last_Y

In [52]:
gold_table_ref = bigquery_client.dataset(dataset_id).table(gold_table_id)
bigquery_client.create_table(gold_table_ref,exists_ok=True)

# Load the transformed DataFrame into the new BigQuery table
job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("Applicant_ID", "STRING"),
        bigquery.SchemaField("Age", "INTEGER"),
        bigquery.SchemaField("Employment_Status", "STRING"),
        bigquery.SchemaField("Income", "NUMERIC"),
        bigquery.SchemaField("Number_of_Dependents", "INTEGER"),
        bigquery.SchemaField("Amount_Requested", "NUMERIC"),
        bigquery.SchemaField("Previous_Assistance_Received", "INTEGER"),
        bigquery.SchemaField("Previous_Assistance_Date", "DATE"),
        bigquery.SchemaField("Supporting_Doc_Verified", "INTEGER"),
        bigquery.SchemaField("Application_Frequency_Last_Year", "INTEGER"),
        bigquery.SchemaField("IP_Address", "STRING"),
        bigquery.SchemaField("Device_Type", "STRING"),
        bigquery.SchemaField("Application_Date", "DATE"),
        bigquery.SchemaField("Fraudulent", "INTEGER"),
        # Add schema fields for one-hot encoded columns
        bigquery.SchemaField("Employment_Employed", "INTEGER"),
        bigquery.SchemaField("Employment_Self-Employed", "INTEGER"),
        bigquery.SchemaField("Employment_Unemployed", "INTEGER"),
        bigquery.SchemaField("Device_Desktop", "INTEGER"),
        bigquery.SchemaField("Device_Mobile", "INTEGER"),
        bigquery.SchemaField("Device_Tablet", "INTEGER"),
        # Add schema fields for age bins
        bigquery.SchemaField("Age_18-24", "INTEGER"),
        bigquery.SchemaField("Age_25-34", "INTEGER"),
        bigquery.SchemaField("Age_35-44", "INTEGER"),
        bigquery.SchemaField("Age_45-54", "INTEGER"),
        bigquery.SchemaField("Age_55-64", "INTEGER"),
        bigquery.SchemaField("Age_65+", "INTEGER"),
        # Add schema fields for calculated columns
        bigquery.SchemaField("Income_to_Amount_Requested", "BIGNUMERIC"),
        bigquery.SchemaField("Time_Since_Previous_Assistance", "INTEGER"),
    ],
    write_disposition="WRITE_TRUNCATE", # Overwrite if table exists
)

# # Select only the columns that exist in the DataFrame and match the schema
# columns_to_load = [field.name for field in gold_table_ref.schema.fields]
# df_to_load = df[columns_to_load]

job = bigquery_client.load_table_from_dataframe(
    df, gold_table_ref, job_config=job_config
)
job.result()  # Wait for the job to complete

print(f"DataFrame loaded into BigQuery table '{gold_table_id}'.")

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/_pandas_helpers.py:535: UserWarning: Pyarrow could not determine the type of columns: Age_Bin.
  warnings.warn(


DataFrame loaded into BigQuery table 'fraud_training_data'.


In [54]:
df_gold = bigquery_client.list_rows(f"{project_id}.{dataset_id}.{gold_table_id}").to_dataframe()
print(df_gold.head())

  Applicant_ID  Age Employment_Status           Income  Number_of_Dependents  \
0        11938   18        Unemployed             0E-9                     5   
1        16205   18        Unemployed             0E-9                     3   
2        16593   18        Unemployed  56222.000000000                     4   
3        21762   18     Self-Employed  45387.000000000                     5   
4        22301   18     Self-Employed             0E-9                     1   

  Amount_Requested  Previous_Assistance_Received  Previous_Assistance_Date  \
0   2844.000000000                             0                      <NA>   
1   7844.000000000                             0                      <NA>   
2    852.000000000                             0                      <NA>   
3   8335.000000000                             0                      <NA>   
4   2312.000000000                             0                      <NA>   

   Supporting_Doc_Verified  Application_Frequency_